<a href="https://colab.research.google.com/github/SohailVibeCoder/IB9AU---GenAI/blob/main/Task14_Visual_RAG_Clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Name:** Sohail Essajee (5757504)

Visual RAG successfully found the right pages for each query — for example, it pulled page 1 (the summary) and page 12 (financials) to correctly identify that AstraZeneca's Product Sales were 55.6B USD (+9%) and Alliance Revenue was $3.1bn (+39%) in FY 2025.

The system works like a real analyst would — instead of searching through raw text, it retrieves the most relevant page, renders it as an image, and uses a vision model to read tables and figures directly, which is especially useful for complex financial layouts.

The approach isn't perfect — for the R&D pipeline question, the model mixed up US and non-US approvals, showing that retrieval quality depends heavily on how well page text matches the query, and the VLM can sometimes misinterpret or over-include information from the image.

In [ ]:
!pip install -q pymupdf sentence-transformers transformers accelerate qwen-vl-utils torchvision Pillow

In [ ]:
import fitz  # PyMuPDF
import torch
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import os
import io

In [ ]:
PDF_PATH = "Full-year-Q4-2025-results-announcement (1).pdf"
assert os.path.exists(PDF_PATH), f"Upload the PDF first! Not found: {PDF_PATH}"

doc = fitz.open(PDF_PATH)
pages = []

MAX_IMG_SIZE = (1024, 1024)  # Resize to avoid GPU OOM

for i, page in enumerate(doc):
    text = page.get_text()
    pix = page.get_pixmap(dpi=150)  # Reduced from 300 to save memory
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    img = img.resize(MAX_IMG_SIZE, Image.LANCZOS)  # Resize for VLM
    pages.append({"page_num": i + 1, "text": text, "image": img})

print(f"Loaded {len(pages)} pages from the PDF.")

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error

In [ ]:
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

page_texts = [p["text"] for p in pages]
page_embeddings = embed_model.encode(page_texts, show_progress_bar=True, convert_to_numpy=True)

print(f"Embedded {len(page_embeddings)} pages, shape: {page_embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedded 39 pages, shape: (39, 384)


In [ ]:
def retrieve_top_pages(query, top_k=1):
    """Retrieve the top-k most relevant pages for a query."""
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    similarities = np.dot(page_embeddings, query_embedding.T).squeeze()
    top_indices = np.argsort(similarities)[::-1][:top_k]
    results = []
    for idx in top_indices:
        results.append({
            "page_num": pages[idx]["page_num"],
            "score": float(similarities[idx]),
            "text_preview": pages[idx]["text"][:200],
            "image": pages[idx]["image"],
        })
    return results

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

print("VLM loaded.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

VLM loaded.


In [ ]:
def visual_rag_query(query, top_k=1):
    """Full Visual RAG pipeline: retrieve page, then ask VLM about the image."""
    retrieved = retrieve_top_pages(query, top_k=top_k)

    for r in retrieved:
        print(f"  Retrieved page {r['page_num']} (score: {r['score']:.4f})")

    images = [r["image"] for r in retrieved]

    image_content = [{"type": "image", "image": img} for img in images]
    messages = [
        {
            "role": "user",
            "content": image_content + [
                {"type": "text", "text": f"Based on the document page(s) shown, answer the following question in detail:\n\n{query}"}
            ],
        }
    ]

    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(vlm_model.device)

    # Clear cache before generation to free up GPU memory
    torch.cuda.empty_cache()

    with torch.no_grad():
        output_ids = vlm_model.generate(**inputs, max_new_tokens=1024)

    generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
    answer = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Clean up to free memory
    del inputs, output_ids, generated_ids
    torch.cuda.empty_cache()

    return answer, retrieved

## Task 1 — Revenue Table Extraction

In [ ]:
query1 = "What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, and how did each change compared to FY 2024?"

print("=" * 80)
print("TASK 1 — Revenue Table Extraction")
print(f"Query: {query1}")
print("=" * 80)

answer1, pages1 = visual_rag_query(query1, top_k=2)

print(f"\nAnswer:\n{answer1}")
print(f"\nRetrieved pages: {[p['page_num'] for p in pages1]}")

TASK 1 — Revenue Table Extraction
Query: What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, and how did each change compared to FY 2024?
  Retrieved page 1 (score: 0.6808)
  Retrieved page 12 (score: 0.6363)

Answer:
According to the document, AstraZeneca's total Product Sales for FY 2025 were $55,573 million, which represented a 9% increase from FY 2024. The Alliance Revenue for FY 2025 was $3,067 million, marking a 39% increase from FY 2024.

Retrieved pages: [1, 12]


## Task 2 — Regional Revenue Breakdown

In [ ]:
query2 = "Which geographic region had the highest Total Revenue growth in FY 2025, and what was the growth rate at constant exchange rates?"

print("=" * 80)
print("TASK 2 — Regional Revenue Breakdown")
print(f"Query: {query2}")
print("=" * 80)

answer2, pages2 = visual_rag_query(query2, top_k=2)

print(f"\nAnswer:\n{answer2}")
print(f"\nRetrieved pages: {[p['page_num'] for p in pages2]}")

TASK 2 — Regional Revenue Breakdown
Query: Which geographic region had the highest Total Revenue growth in FY 2025, and what was the growth rate at constant exchange rates?
  Retrieved page 1 (score: 0.5130)
  Retrieved page 34 (score: 0.5043)

Answer:
According to the document, the geographic region with the highest Total Revenue growth in FY 2025, adjusted for constant exchange rates, was Europe. The growth rate for Europe was 13%.

Retrieved pages: [1, 34]


## Task 3 — R&D Pipeline Interpretation

In [ ]:
query3 = "Which medicines received regulatory approvals in the US between November 2025 and February 2026, and for what indications?"

print("=" * 80)
print("TASK 3 — R&D Pipeline Interpretation")
print(f"Query: {query3}")
print("=" * 80)

answer3, pages3 = visual_rag_query(query3, top_k=2)

print(f"\nAnswer:\n{answer3}")
print(f"\nRetrieved pages: {[p['page_num'] for p in pages3]}")

TASK 3 — R&D Pipeline Interpretation
Query: Which medicines received regulatory approvals in the US between November 2025 and February 2026, and for what indications?
  Retrieved page 13 (score: 0.5111)
  Retrieved page 12 (score: 0.4945)

Answer:
Between November 2025 and February 2026, AstraZeneca received regulatory approvals for several medicines in the US. Here are the details:

1. **DESTINY-Gastric04 (EU)**
   - **Indication:** Locally advanced or metastatic HER2-positive (IHC3+ or IHC2+/ISH+) gastric or gastroesophageal junction adenocarcinoma who have received a prior trastuzumab-based regimen.

2. **DESTINY-Breast09 (US)**
   - **Indication:** Unresectable or metastatic HR-positive, HER2 low (IHC 1+ or IHC 2+/ISH-) or HER2 ultraslow (IHC 0 with membrane staining) breast cancer that has progressed on one or more endocrine therapies in the metastatic setting.

3. **DESTINY-Breast06 (CN)**
   - **Indication:** Unresectable or metastatic HR-positive, HER2 low (IHC 1+ or IHC 2+/ISH